In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import (
    PROJECT_ROOT,
    DATA_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    WEATHER_DIR,
    SOIL_DIR,
    RAW_DIR,
    OUTPUT_DIR,
    PROCESSED_DATASET,
    WEATHER_FEATHER,
    MERGED_DATA_DIR,
    interim_csb_path,
)


# Crop Rotation Strategy Modeling Plan

## 1. Summary of Current Findings

Based on the analysis of CSB (2008-2024) and weather data:

- **Dominant Crops**: Corn and Soybeans are the primary drivers, with specific analysis focused on their acreage trends.
- **Rotation Classification**: Current logic classifies sequences within time windows (e.g., 2008-2015) into "Continuous [Crop]", "Rotation [Crop1]-[Crop2]", or "Complex/Mixed".
- **Acreage Dynamics**:
  - **Persistence**: Crop acreage is highly auto-correlated (Correlation ~0.99 for Corn), indicating strong adherence to rotation plans.
  - **Weather Impact**: Planting precipitation has a negative correlation (-0.165) with Corn acreage, while Growing Degree Days (GDD) have a weak positive effect.
  - **Volatility**: GARCH(1,1) models have been successfully fitted to model the volatility of acreage changes.

## 2. Data Exploration

**Goal**: Visualize and understand the distribution of rotational strategies.

- **Define Target Variable**: Formally define "Rotational Strategy" for each polygon (e.g., "Corn-Soy", "Continuous Corn", "Dairy Rotation", "Fallow-Mix"). This can be done by analyzing the 8-16 year sequence of each polygon.
- **Visualizations**:
  - **Histogram**: Plot the frequency of each defined rotational strategy to check for class imbalance.
  - **2-D Map Heat Map**: Visualize the spatial distribution of dominant strategies across New York state (e.g., color-coded polygons).
  - **Time-Series Plot**: Track the proportion of total acreage dedicated to each strategy over time (e.g., is "Continuous Corn" decreasing?).

In [ ]:
%run src/csb_classification_data_processing.py
%run src/csb_classification_data_exploration.py

hay, grass is related to dairy creature production, can be combined

## 3. Feature Engineering

**Goal**: Create a rich feature set for model training.

- **Static Features (Polygon-Level)**:
  - **Soil Data**: Join existing soil datasets (as identified in `CSB_weather_analysis.ipynb`) to extract attributes like drainage class, soil texture, and slope.
  - **Geometry**: Centroid coordinates (Lat/Lon), area, and perimeter.
- **Dynamic Features (Time-Dependent)**:
  - **Weather Data**: Integrate annual/seasonal metrics (Planting Precip, Growing GDD, anomalies) for each polygon's location.
  - **Plant Cycle Prior**: Encode the crop history as lag features (e.g., `Crop_t-1`, `Crop_t-2`).
  - **Neighbor Information**: Aggregate features from adjacent polygons (e.g., "Dominant neighbor strategy") to capture regional practices.

In [ ]:
# %run src/generate_soil_mapping.py
%run src/feature_engineering.py
%run src/feature_exploration.py

## 4. Model Fitting

**Goal**: Predict the "Rotational Strategy" (classification) or the "Next Crop" (sequence prediction) for each polygon.

- **Baseline / Simple Models**:
  - **KNN (K-Nearest Neighbors)**: Classify strategy based on the $k$ spatially nearest polygons. Good for capturing regional clustering.
  - **XGBoost (Gradient Boosting)**: Train on tabular features (Soil + Weather + History). Handles non-linear interactions and feature importance well.
- **Advanced / Spatio-Temporal Models**:
  - **Geometric RNN (Spatio-Temporal GNN)**:
    - **Graph Construction**: Build a graph where nodes are polygons and edges connect spatial neighbors.
    - **Architecture**: Use a Recurrent Neural Network (LSTM/GRU) to process the temporal sequence of crops/weather, augmented with Graph Convolutional Layers (GCN/GAT) to pool information from spatial neighbors at each time step.
    - **Objective**: Predict the crop choice for the next year or classify the underlying long-term strategy.

In [ ]:
%run src/model_baseline.py

## MODEL EVALUATION AND RESULTS

### Model Training Setup

**Dataset Configuration:**
- Sample Size: 50,000 records (from 5.9M total for computational efficiency)
- Train/Test Split: ≤2022 for training (43,804 samples), >2022 for testing (6,196 samples)
- Target Classes: 5-class crop classification (Corn, Soybeans, Alfalfa, Combined Hay/Grass, Other)
- Class Distribution: Imbalanced dataset with Combined Hay/Grass (56%) dominating

**Feature Engineering:**
- **Geometric Features**: Longitude_Norm, Latitude_Norm, County-level aggregations
- **Temporal Features**: Crop history lags (Lag1, Lag2) 
- **Environmental Features**: Weather data (Planting_Precip, Growing_GDD) with temporal lags
- **Preprocessing**: StandardScaler for KNN, balanced class weights for CatBoost

**Model Configurations:**
1. **K-Nearest Neighbors (KNN)**:
   - Parameters: n_neighbors=5, StandardScaler preprocessing
   - Spatial approach: Uses coordinate distance for geographic clustering
   - Stratified subsampling: 30,000 training samples for efficiency

2. **CatBoost**:
   - Parameters: 100 iterations, depth=4, learning_rate=0.1, auto_class_weights='Balanced'
   - Tree-based approach: Hierarchical spatial decision trees
   - Built-in categorical feature handling

### Performance Analysis

**Quantitative Results:**
| Model | Accuracy | Precision (Macro) | Recall (Macro) | F1-Score (Macro) | ROC AUC (Macro) |
|-------|----------|------------------|----------------|------------------|-----------------|
| **KNN** | **69.48%** | 53.14% | 50.56% | 50.34% | 80.40% |
| **CatBoost** | 67.33% | 53.08% | **60.16%** | **53.93%** | **86.64%** |

**Key Performance Insights:**

1. **Overall Performance**:
   - **KNN achieves higher accuracy** (69.48%) due to effective spatial clustering
   - **CatBoost shows better macro recall** (60.16%) and F1-score (53.93%), indicating improved minority class performance
   - **CatBoost dominates ROC AUC** (86.64%), suggesting superior probability calibration

2. **Class-Specific Performance**:
   - **Combined Hay/Grass**: Both models excel (F1 ~85%), leveraging the dominant class advantage
   - **Corn**: KNN performs better (F1: 58.45% vs 51.09%), benefiting from spatial proximity patterns
   - **Soybeans & Alfalfa**: CatBoost shows improvement for minority classes through balanced weighting
   - **Other category**: Remains challenging for both models (~25-35% F1)

3. **Model Behavior Analysis**:
   
   **KNN Strengths:**
   - **Spatial Autocorrelation**: Excellent at capturing "neighboring fields → similar crops" patterns
   - **Regional Clustering**: Effective for geographically concentrated crop types
   - **High Precision for Dominant Class**: 89% precision for Hay/Grass
   
   **CatBoost Strengths:**
   - **Minority Class Recovery**: Better recall for Soybeans (71%) and Alfalfa (74%)
   - **Feature Interaction Learning**: Captures complex location × weather × history interactions
   - **Probability Calibration**: Superior ROC AUC indicates better uncertainty quantification

### Geographic Feature Integration Impact

**Feature Importance Analysis (CatBoost):**
- **Geometric features highlighted in red** demonstrate significant contribution to model performance
- **Multi-scale spatial modeling** successfully integrates field-level coordinates with county-level aggregations
- **Spatial context enhancement** provides the missing link between environmental conditions and regional agricultural practices

**Conclusion:**
Both models successfully leverage geometric features for crop classification, with KNN excelling at spatial clustering and CatBoost providing better minority class performance through sophisticated feature interactions. The 80%+ ROC AUC scores indicate strong predictive capability, while the ~69% accuracy represents reasonable performance for this complex agricultural decision modeling task with inherent uncertainty in farmer behavior.

# 5. Neural network model

In [ ]:
import torch
import numpy as np

# Geometric CNN and RNN Models for Crop Classification
import numpy as np
import torch

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Model Hyperparameters
# =====================================
input_size = 10          # Number of input features (auto-adjusted based on data)
hidden_size = 32         # Number of hidden units in neural networks
learning_rate = 0.0001   # Learning rate for optimization
num_epochs = 1500          # Number of training epochs
dropout = 0.2            # Dropout probability for regularization
batch_size = 64          # Batch size for training
gradient_clip = 1.0      # Gradient clipping threshold
sample_size = 10000      # Data sample size for computational efficiency
k_neighbors = 5          # Number of spatial neighbors for CNN adjacency
sequence_length = 3      # Temporal sequence length for RNN
num_layers = 2           # Number of layers in both CNN and RNN

print('Hyperparameters configured:')
print(f'  Input Size: {input_size}')
print(f'  Hidden Size: {hidden_size}')
print(f'  Learning Rate: {learning_rate}')
print(f'  Epochs: {num_epochs}')
print(f'  Dropout: {dropout}')
print(f'  Batch Size: {batch_size}')
print(f'  Gradient Clip: {gradient_clip}')

# Run the geometric deep learning models
print('\nStarting Geometric Deep Learning Training...')
%run src/model_advanced.py